In [1]:
import cv2
import numpy as np
from didascalie.com import register_seg, serve
from fundus_lesions_toolkit.constants import LESIONS
from fundus_lesions_toolkit.models import segment
from fundus_odmac_toolkit.models.segmentation import segment as segment_odmac

In [ ]:
@register_seg("fundus_lesions_toolkit")
def lesion_segment(image):
    h, w, c = image.shape
    image = cv2.resize(image, (1024, 1024), interpolation=cv2.INTER_AREA)
    # One map per class; the first is the background.
    segmentation = (
        segment(image, autofit_resolution=False, reverse_autofit=False)[1:]
        .cpu()
        .numpy()
    )
    # Back to the frame's resolution.
    segmentations = [
        cv2.resize(
            segmentation[i].astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST
        )
        for i in range(len(segmentation))
    ]
    return {lesion: mask for lesion, mask in zip(LESIONS[1:], segmentations)}


@register_seg("fundus_odmac_toolkit")
def odmac_segment(image):
    h, w, c = image.shape
    image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_AREA)
    segmentation = (
        segment_odmac(image, autofit_resolution=False, reverse_autofit=False)
        .cpu()
        .argmax(0)
        .numpy()
    )

    # Back to the frame's resolution.
    segmentation = cv2.resize(
        segmentation.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST
    )

    return {"OD": segmentation == 1, "Macula": segmentation == 2}


serve()


[didascalie] listening on tcp://*:5556, registered: ['fundus_lesions_toolkit', 'fundus_odmac_toolkit']
